# Mount HiRes Elevation Axis Tracking vs Atmospheric Seeing

## Overview
Analysis of the relationship between telescope elevation axis standard deviation (tracking error) and atmospheric seeing ellipticity over time.

This notebook examines temporal trends and correlations between:
- **Elevation Axis Stability**: RMS error in elevation (altitude) axis tracking
- **Seeing Ellipticity**: Atmospheric distortion measured by WFS systems

Multiple visualization approaches are presented to identify optimal data presentation methods.

In [1]:
# Section 1: Import Required Libraries
from datetime import datetime
from pathlib import Path

import numpy as np
import scipy
import pandas as pd
import matplotlib
matplotlib.use('agg')
from matplotlib import style
style.use('ggplot')
import matplotlib.pyplot as plt
import seaborn as sns

%load_ext autoreload
%autoreload 2

import sqlalchemy
from sqlalchemy import text
import pymysql
from pymysql.cursors import DictCursor

# For statistical analysis
from scipy import stats
from scipy.stats import linregress, gaussian_kde

# For FITS file handling (if needed)
try:
    from astropy.io import fits
    import astropy.units as u
except ImportError:
    print("astropy not available - will work with CSV/database only")

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Section 2: Load and Prepare Data
# This section loads data from the mount_hires database (rd_stats and rd_meta tables)
# and WFS seeing data from the measurements database

# Note: Update credentials in my_secret.py
try:
    import my_secret as secret
    host_hires = secret.host_hires_logs
    db_hires = secret.database_hires_logs
    user_hires = secret.usr_hires_logs
    pwd_hires = secret.pwd_hires_logs
    
    host_bg = secret.host_background_logs
    db_bg = secret.database_background_logs
    user_bg = secret.usr_background_logs
    pwd_bg = secret.pwd_background_logs
    
    # Create SQLAlchemy engines
    engine_hires = sqlalchemy.create_engine(
        f"mysql+pymysql://{user_hires}:{pwd_hires}@{host_hires}/{db_hires}"
    )
    engine_bg = sqlalchemy.create_engine(
        f"mysql+pymysql://{user_bg}:{pwd_bg}@{host_bg}/{db_bg}"
    )
    
    print("Database connections established")
    
except Exception as e:
    print(f"Error loading database credentials: {e}")
    print("Will attempt to use mock data or CSV files instead")

In [ ]:
# Query elevation tracking statistics from rd_stats table
# rd_stats contains: mean, std, min, max, RMS, range for telescope_alterr in arc-seconds

try:
    query_stats = """
    SELECT 
        filename,
        MST_time,
        param,
        std,
        mean,
        rms,
        min,
        max,
        p50 as median
    FROM rd_stats 
    WHERE param LIKE '%alterr%'
    AND MST_time >= DATE_SUB(NOW(), INTERVAL 6 MONTH)
    ORDER BY MST_time
    """
    
    df_alt_stats = pd.read_sql(query_stats, engine_hires)
    df_alt_stats['MST_time'] = pd.to_datetime(df_alt_stats['MST_time'])
    df_alt_stats.set_index('MST_time', inplace=True)
    
    print(f"Loaded {len(df_alt_stats)} elevation tracking statistics records")
    print(f"Date range: {df_alt_stats.index.min()} to {df_alt_stats.index.max()}")
    print(f"\nElevation Statistics Summary:")
    print(df_alt_stats.describe())
    
except Exception as e:
    print(f"Error loading elevation stats: {e}")
    df_alt_stats = None

In [ ]:
# Query seeing data from background logs (measurements database)
# WFS seeing measurements from various instruments

try:
    query_seeing = """
    SELECT 
        timestamp,
        FROM_UNIXTIME(timestamp/1000) as time_mst,
        value as seeing_arcsec
    FROM measurements.wfs_seeing
    WHERE timestamp >= UNIX_TIMESTAMP(DATE_SUB(NOW(), INTERVAL 6 MONTH)) * 1000
    ORDER BY timestamp
    """
    
    df_seeing = pd.read_sql(query_seeing, engine_bg)
    df_seeing['time_mst'] = pd.to_datetime(df_seeing['time_mst'])
    df_seeing.set_index('time_mst', inplace=True)
    
    # Filter for reasonable seeing values (0.2 to 3.0 arcsec)
    df_seeing = df_seeing[(df_seeing['seeing_arcsec'] > 0.2) & (df_seeing['seeing_arcsec'] < 3.0)]
    
    print(f"Loaded {len(df_seeing)} WFS seeing measurements")
    print(f"Date range: {df_seeing.index.min()} to {df_seeing.index.max()}")
    print(f"\nWFS Seeing Summary:")
    print(df_seeing.describe())
    
except Exception as e:
    print(f"Error loading seeing data: {e}")
    df_seeing = None

In [ ]:
# Section 3: Calculate Seeing Ellipticity
# Seeing ellipticity can be computed from:
# 1. Zernike coefficient differences (astigmatism C_2_2)
# 2. FWHM in orthogonal directions
# Here we compute from available WFS data

try:
    # Query Zernike astigmatism data (if available in rd_psd or custom table)
    # For now, we'll create a proxy ellipticity metric from seeing variations
    query_raw_seeing = """
    SELECT 
        timestamp,
        FROM_UNIXTIME(timestamp/1000) as time_mst,
        value as raw_seeing_arcsec
    FROM measurements.wfs_raw_seeing
    WHERE timestamp >= UNIX_TIMESTAMP(DATE_SUB(NOW(), INTERVAL 6 MONTH)) * 1000
    ORDER BY timestamp
    """
    
    df_raw_seeing = pd.read_sql(query_raw_seeing, engine_bg)
    df_raw_seeing['time_mst'] = pd.to_datetime(df_raw_seeing['time_mst'])
    df_raw_seeing.set_index('time_mst', inplace=True)
    df_raw_seeing = df_raw_seeing[(df_raw_seeing['raw_seeing_arcsec'] > 0.2) & 
                                   (df_raw_seeing['raw_seeing_arcsec'] < 4.0)]
    
    print(f"Loaded {len(df_raw_seeing)} raw WFS seeing measurements")
    
    # Compute ellipticity as ratio of raw_seeing to corrected seeing
    # This indicates the effectiveness of WFS correction
    # Higher ratio = more atmospheric distortion (higher ellipticity)
    
    if df_seeing is not None and df_raw_seeing is not None:
        # Merge on nearest timestamp
        df_combined = pd.concat([df_seeing, df_raw_seeing], axis=1)
        df_combined['ellipticity'] = df_combined['raw_seeing_arcsec'] / (df_combined['seeing_arcsec'] + 0.01)
        df_combined['ellipticity'] = df_combined['ellipticity'].clip(1.0, None)  # Ratio >= 1.0
        
        print(f"\nSeeing Ellipticity Summary:")
        print(df_combined[['ellipticity']].describe())
    else:
        print("Cannot compute ellipticity - missing seeing data")
        df_combined = None
        
except Exception as e:
    print(f"Error calculating seeing ellipticity: {e}")
    df_combined = None

In [ ]:
# Section 4: Compute Elevation Axis Standard Deviation
# Extract tracking error statistics for elevation (altitude) axis from rd_stats

if df_alt_stats is not None:
    # Filter for telescope_alterr_arcsec (corrected elevation error)
    df_alt_std = df_alt_stats[df_alt_stats['param'] == 'telescope_alterr_arcsec'].copy()
    
    if len(df_alt_std) > 0:
        # Rename columns for clarity
        df_alt_std.columns = df_alt_std.columns.str.lower()
        
        print(f"Elevation Axis Tracking Statistics:")
        print(f"  Standard Deviation - Min: {df_alt_std['std'].min():.3f}\", Max: {df_alt_std['std'].max():.3f}\"")
        print(f"  RMS Error - Min: {df_alt_std['rms'].min():.3f}\", Max: {df_alt_std['rms'].max():.3f}\"")
        print(f"  Mean Error - Min: {df_alt_std['mean'].min():.3f}\", Max: {df_alt_std['mean'].max():.3f}\"")
        
        # Use RMS as the tracking error metric (most representative of servo performance)
        tracking_error_metric = 'rms'  # Can also use 'std' or 'mean'
        
        print(f"\nUsing '{tracking_error_metric}' as elevation tracking error metric")
        print(f"Data points: {len(df_alt_std)}")
    else:
        print("No elevation statistics found")
        df_alt_std = None
else:
    print("Elevation statistics data not available")
    df_alt_std = None

In [ ]:
# Section 5: Resample Data by Time Period
# Resample to daily, weekly, and monthly bins to reduce noise and identify trends

def resample_data(df_alt, df_seeing_ellip, resample_period='D'):
    """
    Resample altitude tracking error and seeing ellipticity to specified period
    
    Parameters:
    - resample_period: 'D' (daily), 'W' (weekly), 'MS' (monthly start)
    """
    
    # Prepare altitude data
    if df_alt is not None and len(df_alt) > 0:
        df_alt_resampled = df_alt[['rms']].resample(resample_period).agg({
            'rms': ['mean', 'std', 'min', 'max', 'count']
        })
        df_alt_resampled.columns = ['alt_rms_mean', 'alt_rms_std', 'alt_rms_min', 'alt_rms_max', 'alt_count']
    else:
        df_alt_resampled = None
    
    # Prepare seeing data
    if df_seeing_ellip is not None and len(df_seeing_ellip) > 0:
        if 'seeing_arcsec' in df_seeing_ellip.columns:
            df_seeing_resampled = df_seeing_ellip[['seeing_arcsec']].resample(resample_period).agg({
                'seeing_arcsec': ['mean', 'std', 'min', 'max', 'count']
            })
            df_seeing_resampled.columns = ['seeing_mean', 'seeing_std', 'seeing_min', 'seeing_max', 'seeing_count']
        else:
            df_seeing_resampled = None
    else:
        df_seeing_resampled = None
    
    # Combine into single dataframe
    if df_alt_resampled is not None and df_seeing_resampled is not None:
        df_combined_resampled = pd.concat([df_alt_resampled, df_seeing_resampled], axis=1)
        # Remove rows with insufficient data
        df_combined_resampled = df_combined_resampled[(df_combined_resampled['alt_count'] >= 5) & 
                                                       (df_combined_resampled['seeing_count'] >= 5)]
    elif df_alt_resampled is not None:
        df_combined_resampled = df_alt_resampled
    else:
        df_combined_resampled = None
    
    return df_combined_resampled

# Create resampled datasets
if df_alt_std is not None and df_seeing is not None:
    df_daily = resample_data(df_alt_std, df_seeing, 'D')
    df_weekly = resample_data(df_alt_std, df_seeing, 'W')
    df_monthly = resample_data(df_alt_std, df_seeing, 'MS')
    
    print("Daily resampled data points:", len(df_daily))
    print("Weekly resampled data points:", len(df_weekly))
    print("Monthly resampled data points:", len(df_monthly))
    
    print("\nDaily Summary:")
    if df_daily is not None:
        print(df_daily.head())
else:
    print("Insufficient data for resampling")
    df_daily = None
    df_weekly = None
    df_monthly = None

In [ ]:
# Section 6: Create Scatter Plot with Trend Analysis
# Generate scatter plot of elevation standard deviation vs seeing with regression line

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Create scatter plots for daily, weekly, and monthly data
resample_sets = [('Daily', df_daily), ('Weekly', df_weekly), ('Monthly', df_monthly)]

for idx, (label, df) in enumerate(resample_sets):
    if df is not None and len(df) > 10:
        ax = axes[idx]
        
        # Extract data
        x = df['seeing_mean'].values
        y = df['alt_rms_mean'].values
        
        # Remove NaN values
        mask = ~(np.isnan(x) | np.isnan(y))
        x = x[mask]
        y = y[mask]
        
        if len(x) > 3:
            # Calculate correlation
            correlation = np.corrcoef(x, y)[0, 1]
            
            # Linear regression
            slope, intercept, r_value, p_value, std_err = linregress(x, y)
            line_x = np.array([x.min(), x.max()])
            line_y = slope * line_x + intercept
            
            # Scatter plot with regression line
            ax.scatter(x, y, alpha=0.6, s=50, edgecolors='k', linewidth=0.5)
            ax.plot(line_x, line_y, 'r--', linewidth=2, label=f'Linear fit (R²={r_value**2:.3f})')
            
            # Labels and title
            ax.set_xlabel('Seeing Ellipticity (arcsec)', fontsize=11, fontweight='bold')
            ax.set_ylabel('Elevation Axis RMS Error (arcsec)', fontsize=11, fontweight='bold')
            ax.set_title(f'{label} Data (n={len(x)})\nCorrelation: {correlation:.3f}, p-value: {p_value:.2e}', 
                        fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=10)
            
            # Print statistics
            print(f"\n{label} Scatter Statistics:")
            print(f"  Correlation coefficient: {correlation:.4f}")
            print(f"  Slope: {slope:.4f} arcsec/arcsec")
            print(f"  Intercept: {intercept:.4f} arcsec")
            print(f"  R-squared: {r_value**2:.4f}")
            print(f"  P-value: {p_value:.2e}")
        else:
            ax.text(0.5, 0.5, f'Insufficient data\n(n={len(x)})', 
                   ha='center', va='center', transform=ax.transAxes)
    else:
        axes[idx].text(0.5, 0.5, f'No {label} data available', 
                      ha='center', va='center', transform=axes[idx].transAxes)

plt.tight_layout()
plt.savefig('scatter_alt_vs_seeing.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Scatter plot saved as 'scatter_alt_vs_seeing.png'")

In [ ]:
# Section 7: Generate Time Series Visualization
# Create dual-axis time series plots showing trends over time

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Daily time series with dual axes
if df_daily is not None and len(df_daily) > 5:
    ax1 = axes[0]
    ax1_twin = ax1.twinx()
    
    # Plot elevation RMS on left axis
    line1 = ax1.plot(df_daily.index, df_daily['alt_rms_mean'], 'b-', linewidth=2, label='Elevation RMS Error')
    ax1.fill_between(df_daily.index, 
                      df_daily['alt_rms_mean'] - df_daily['alt_rms_std'],
                      df_daily['alt_rms_mean'] + df_daily['alt_rms_std'],
                      alpha=0.2, color='blue', label='RMS ±1σ')
    ax1.set_xlabel('Date', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Elevation Axis Error (arcsec)', fontsize=11, fontweight='bold', color='b')
    ax1.tick_params(axis='y', labelcolor='b')
    ax1.grid(True, alpha=0.3)
    
    # Plot seeing on right axis
    if 'seeing_mean' in df_daily.columns:
        line2 = ax1_twin.plot(df_daily.index, df_daily['seeing_mean'], 'r-', linewidth=2, label='Atmospheric Seeing')
        ax1_twin.fill_between(df_daily.index,
                             df_daily['seeing_mean'] - df_daily['seeing_std'],
                             df_daily['seeing_mean'] + df_daily['seeing_std'],
                             alpha=0.2, color='red', label='Seeing ±1σ')
        ax1_twin.set_ylabel('Atmospheric Seeing (arcsec)', fontsize=11, fontweight='bold', color='r')
        ax1_twin.tick_params(axis='y', labelcolor='r')
    
    ax1.set_title('Daily Trends: Elevation Axis RMS vs Atmospheric Seeing', 
                  fontsize=12, fontweight='bold')
    
    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax1_twin.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

# Weekly time series
if df_weekly is not None and len(df_weekly) > 3:
    ax2 = axes[1]
    ax2_twin = ax2.twinx()
    
    line1 = ax2.plot(df_weekly.index, df_weekly['alt_rms_mean'], 'b-o', linewidth=2, markersize=6, label='Elevation RMS')
    ax2.fill_between(df_weekly.index,
                     df_weekly['alt_rms_mean'] - df_weekly['alt_rms_std'],
                     df_weekly['alt_rms_mean'] + df_weekly['alt_rms_std'],
                     alpha=0.2, color='blue')
    ax2.set_xlabel('Date', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Elevation Axis Error (arcsec)', fontsize=11, fontweight='bold', color='b')
    ax2.tick_params(axis='y', labelcolor='b')
    ax2.grid(True, alpha=0.3)
    
    if 'seeing_mean' in df_weekly.columns:
        line2 = ax2_twin.plot(df_weekly.index, df_weekly['seeing_mean'], 'r-o', linewidth=2, markersize=6, label='Seeing')
        ax2_twin.fill_between(df_weekly.index,
                             df_weekly['seeing_mean'] - df_weekly['seeing_std'],
                             df_weekly['seeing_mean'] + df_weekly['seeing_std'],
                             alpha=0.2, color='red')
        ax2_twin.set_ylabel('Atmospheric Seeing (arcsec)', fontsize=11, fontweight='bold', color='r')
        ax2_twin.tick_params(axis='y', labelcolor='r')
    
    ax2.set_title('Weekly Trends: Elevation Axis RMS vs Atmospheric Seeing',
                  fontsize=12, fontweight='bold')
    
    lines1, labels1 = ax2.get_legend_handles_labels()
    lines2, labels2 = ax2_twin.get_legend_handles_labels()
    ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('timeseries_alt_vs_seeing.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Time series plot saved as 'timeseries_alt_vs_seeing.png'")

In [ ]:
# Section 8: Produce Hexbin Density Plot
# Hexbin visualization to show density distribution and identify clusters

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

resample_sets = [('Weekly', df_weekly), ('Monthly', df_monthly)]

for idx, (label, df) in enumerate(resample_sets):
    if df is not None and len(df) > 5:
        ax = axes[idx]
        
        x = df['seeing_mean'].values
        y = df['alt_rms_mean'].values
        
        # Remove NaN values
        mask = ~(np.isnan(x) | np.isnan(y))
        x = x[mask]
        y = y[mask]
        
        if len(x) > 5:
            # Create hexbin plot
            hb = ax.hexbin(x, y, gridsize=8, cmap='YlOrRd', mincnt=1, edgecolors='black', linewidths=0.2)
            
            # Add regression line
            slope, intercept, r_value, p_value, std_err = linregress(x, y)
            line_x = np.array([x.min(), x.max()])
            line_y = slope * line_x + intercept
            ax.plot(line_x, line_y, 'b--', linewidth=2, label=f'Linear fit (R²={r_value**2:.3f})')
            
            ax.set_xlabel('Atmospheric Seeing (arcsec)', fontsize=11, fontweight='bold')
            ax.set_ylabel('Elevation Axis RMS Error (arcsec)', fontsize=11, fontweight='bold')
            ax.set_title(f'{label} Data Density (n={len(x)})', fontsize=11, fontweight='bold')
            ax.legend(fontsize=10)
            
            # Colorbar
            cb = plt.colorbar(hb, ax=ax)
            cb.set_label('Count', fontsize=10)
            
            print(f"\n{label} Hexbin Statistics:")
            print(f"  Bins with data: {np.sum(hb.get_array() > 0)}")
            print(f"  Max count per bin: {np.max(hb.get_array())}")
        else:
            ax.text(0.5, 0.5, f'Insufficient data (n={len(x)})',
                   ha='center', va='center', transform=ax.transAxes)
    else:
        axes[idx].text(0.5, 0.5, f'No {label} data available',
                      ha='center', va='center', transform=axes[idx].transAxes)

plt.tight_layout()
plt.savefig('hexbin_alt_vs_seeing.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Hexbin density plot saved as 'hexbin_alt_vs_seeing.png'")

## Recommendations for Optimal Data Presentation

### Best Practices for Visualizing Elevation Axis Performance vs Atmospheric Seeing

#### 1. **Primary Figure: Dual-Axis Time Series (Recommended for SPIE Paper)**
   - **Why it's best**: Shows temporal evolution of both parameters simultaneously
   - **Advantages**:
     - Reveals causality and response time (tracking error response to seeing changes)
     - Enables audience to correlate performance degradation with atmospheric conditions
     - Shows trends over extended observation periods
   - **Best practice**: Use weekly resampling for clarity without losing important variations
   - **Visual enhancement**: Use fill_between to show ±1σ uncertainty bands

#### 2. **Supporting Figure: Scatter Plot with Regression Analysis**
   - **Why include it**: Quantifies the linear relationship strength
   - **Key metrics to report**:
     - Correlation coefficient (r)
     - R² value (coefficient of determination)
     - P-value (statistical significance)
     - Slope (performance degradation rate per arcsec of seeing)
   - **Best resolution**: Weekly or monthly aggregation to reduce scatter
   - **Interpretation**: 
     - Strong correlation (r > 0.7) suggests atmospheric seeing is dominant factor
     - Weak correlation (r < 0.3) suggests other mechanical/control factors dominate

#### 3. **Supplementary Figure: Hexbin Density Plot**
   - **Why useful**: Reveals operational regimes and clustering
   - **Advantages**:
     - Identifies "sweet spots" where performance is optimal
     - Shows data concentration in parameter space
     - Highlights outliers or exceptional conditions
   - **Use case**: Technical audience or detailed analysis sections

#### 4. **Alternative: Contour Plot (Monthly Aggregation)**
   - **When to use**: If you have dense grid of data
   - **Advantages**: Shows smooth gradients in performance space
   - **Less recommended for**: Sparse or irregularly distributed data

### Specific Recommendations for SPIE 2026 Paper

**For Figure 1 (Main Results):**
- Use the **dual-axis weekly time series** showing 3-6 months of data
- Include clear labeling of extreme events (high seeing or poor tracking)
- Add annotation boxes highlighting key correlations or anomalies

**For Supporting Material:**
- Include the **scatter plot with statistics** showing correlation analysis
- Optional: Monthly-aggregated data for smoother trends

**Data Filtering Recommendations:**
- Exclude tracking data when telescope was slewing (not tracking)
- Filter seeing data: 0.2 - 3.0 arcsec (physical measurement limits)
- Minimum 5 observations per time bin (daily/weekly/monthly)
- Consider only "science" observation modes, exclude calibration data

### Key Metrics to Highlight

| Metric | Interpretation | Target |
|--------|---------------|----|
| **Correlation (r)** | Strength of seeing-performance link | > 0.5 for good correlation |
| **Slope** | arcsec error per arcsec seeing | Characteristic of servo system |
| **R²** | Fraction of variance explained by seeing | > 0.25 for meaningful relationship |
| **Baseline RMS** | Performance in excellent seeing | Design specification compliance |

### Color Scheme Recommendations
- **Elevation tracking error**: Blue (authority), consistent with technical plots
- **Seeing/atmospheric**: Red/orange (warning, external factor)
- **Confidence bands**: Lighter tints of corresponding colors
- **Regression line**: Contrasting color (dashed, for distinction from data)

In [ ]:
# BONUS: 2D Kernel Density Estimation (Advanced Visualization)
# Creates smooth density contours for publication-quality figures

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

resample_sets = [('Weekly', df_weekly, 8), ('Monthly', df_monthly, 5)]

for idx, (label, df, gridsize) in enumerate(resample_sets):
    if df is not None and len(df) > 10:
        ax = axes[idx]
        
        x = df['seeing_mean'].values
        y = df['alt_rms_mean'].values
        
        mask = ~(np.isnan(x) | np.isnan(y))
        x = x[mask]
        y = y[mask]
        
        if len(x) > 10:
            # Compute 2D kernel density
            xy = np.vstack([x, y])
            z = gaussian_kde(xy)(xy)
            
            # Sort by density so densest points are plotted last
            idx_sort = np.argsort(z)
            x_sort, y_sort, z_sort = x[idx_sort], y[idx_sort], z[idx_sort]
            
            # Scatter plot colored by density
            scatter = ax.scatter(x_sort, y_sort, c=z_sort, s=60, cmap='viridis', 
                               edgecolors='k', linewidth=0.5, alpha=0.7)
            
            # Add regression line
            slope, intercept, r_value, _, _ = linregress(x, y)
            line_x = np.array([x.min(), x.max()])
            line_y = slope * line_x + intercept
            ax.plot(line_x, line_y, 'r--', linewidth=2.5, label=f'Linear fit (R²={r_value**2:.3f})')
            
            ax.set_xlabel('Atmospheric Seeing (arcsec)', fontsize=11, fontweight='bold')
            ax.set_ylabel('Elevation Axis RMS Error (arcsec)', fontsize=11, fontweight='bold')
            ax.set_title(f'{label} Data with Density (n={len(x)})', fontsize=11, fontweight='bold')
            ax.legend(fontsize=10)
            ax.grid(True, alpha=0.3)
            
            cb = plt.colorbar(scatter, ax=ax)
            cb.set_label('Density', fontsize=10)
        else:
            ax.text(0.5, 0.5, f'Insufficient data (n={len(x)})',
                   ha='center', va='center', transform=ax.transAxes)
    else:
        axes[idx].text(0.5, 0.5, f'No {label} data', ha='center', va='center',
                      transform=axes[idx].transAxes)

plt.tight_layout()
plt.savefig('kde_density_alt_vs_seeing.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ 2D KDE density plot saved as 'kde_density_alt_vs_seeing.png'")

## Summary: Figure Selection Guide for SPIE 2026 Paper

### Quick Reference Table

| Figure Type | Best For | Data Aggregation | Space Req'd | Impact |
|---|---|---|---|---|
| **Dual-Axis Time Series** | ⭐ PRIMARY | Weekly | 1 full page | High - Shows trends & correlation |
| **Scatter + Regression** | ⭐ SECONDARY | Monthly | 0.5 page | High - Quantifies relationship |
| **Hexbin Density** | Supporting | Weekly/Monthly | 0.5 page | Medium - Shows operational regimes |
| **2D KDE Contours** | Optional | Weekly/Monthly | 0.5 page | Medium - Publication quality |

### Implementation Checklist for Publication

- [ ] Extract last 6-12 months of:
  - Elevation axis RMS error (rd_stats table)
  - Atmospheric seeing measurements (WFS data)
  
- [ ] Apply quality filters:
  - Tracking mode only (exclude slewing)
  - Seeing range: 0.2-3.0 arcsec
  - Minimum 5 observations per time bin
  
- [ ] Generate visualizations:
  - [ ] Weekly time series (PRIMARY)
  - [ ] Monthly scatter plot (SECONDARY)
  - [ ] Optional: KDE density plot
  
- [ ] Calculate and report:
  - [ ] Pearson correlation coefficient
  - [ ] Linear regression slope & R²
  - [ ] P-value for statistical significance
  
- [ ] Document findings:
  - [ ] Performance baseline in excellent seeing
  - [ ] Performance degradation rate
  - [ ] Dominant factors limiting tracking

### Technical Notes

**For Python/Matplotlib Optimization:**
```python
# High-DPI figure for publication
plt.savefig('figure.png', dpi=300, bbox_inches='tight')

# Use seaborn for better defaults
sns.set_style("whitegrid")
sns.set_palette("husl")
```

**Database Query Optimization:**
- Index on (filename, param, MST_time) for rd_stats table
- Filter by time range in query (not in Python)
- Use `read_sql_query` with WHERE clauses for efficiency

**Statistical Considerations:**
- If r < 0.3: weak correlation suggests external factors dominate servo performance
- If r > 0.7: strong correlation suggests seeing is limiting factor
- Include confidence intervals (error bars) on all data points